# Uji Protokol Tesis: baseline Multi-GIN+EU dan MEGA-GIN (seed 1)

Tahap 2 rencana tesis. Menjalankan **baseline** kedua backbone dengan **protokol tesis** (`configs/thesis/`), bukan protokol paper:

| | Paper MEGA | Protokol tesis |
|---|---|---|
| 1 epoch | 256 iterasi (~1/6 data latih) | 1 putaran penuh seed edge latih (HI 1.587 / LI 2.165 iterasi) |
| Epoch maksimum | 400 | 20 |
| Evaluasi | val + test tiap 4 epoch | **val** tiap epoch |
| Pemilihan model | argmax F1 val | argmax F1 val + **early stopping** (5 evaluasi tanpa kenaikan, dihitung setelah F1 val > 0) |
| Test | tiap evaluasi | **sekali**, dengan checkpoint val terbaik |
| Output tambahan | - | `run_summary.json` (epoch terbaik, waktu, memori GPU, cakupan prediksi), skor per transaksi `scores_val.npz` / `scores_test.npz` |

Model, optimizer, dan hyperparameter sama persis dengan config paper.

**Tujuan uji ini:** (1) memastikan protokol berjalan di Kaggle, (2) melihat kapan model berhenti (rata-rata epoch -> anggaran 110 run), (3) membandingkan F1 baseline dengan paper sebagai pengganti Tahap C.

**Cara menjalankan** (perkiraan maksimal tanpa early stopping: HI ~3,5 jam untuk 2 model, LI ~5 jam):
1. Settings: Input dataset IBM AML, **GPU T4**, **Internet On**.
2. Pilih `DATASET` di sel "Jalankan run" (`"Small-HI"` dulu, lalu `"Small-LI"` di sesi berikutnya).
3. Jalankan di latar belakang: **Save Version -> Save & Run All (Commit)**. Hasil tersimpan di Output versi itu.
4. Unduh `results_thesis.zip` dari Output dan simpan ke `project_aml/hasil_kaggle/protokol/`.

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/aanpujianto070990/MEGA-GNN-TMLR.git"
REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

assert os.path.isdir(f"{REPO_DIR}/MegaGNN"), "Clone gagal -- pastikan Settings -> Internet = On, lalu jalankan ulang sel ini."
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
import sys, torch

print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))

TORCH_MM = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
CUDA_TAG = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
PYG_WHL = f"https://data.pyg.org/whl/torch-{TORCH_MM}+{CUDA_TAG}.html"

!pip install -q torch_geometric yacs ogb gdown tensorboardX torchmetrics pybind11
!pip install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {PYG_WHL}
# genagg sengaja non-editable (lihat Pilot_Reproduksi_Kaggle.ipynb).
!pip uninstall -y -q genagg
!pip install -q --no-deps {REPO_DIR}/genagg

!cd {REPO_DIR}/MegaGNN/cpp && python setup.py build_ext --inplace > build.log 2>&1; ls *.so
!cd {REPO_DIR} && python -c "import MegaGNN, torch_scatter, torch_sparse; from MegaGNN.cpp import ports_cpp; print('Import MegaGNN OK')"

## 2. Data

Data olahan dibangun otomatis pada run pertama tiap dataset (cache lama tanpa atribut tesis `edge_id`/`delta_t` dibangun ulang sendiri oleh `AMLDataset`).

In [ ]:
import glob

DATA_DIR = "/tmp/aml_data"
OUT_DIR = "/kaggle/working/results_thesis"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = {"Small-HI": "HI-Small_Trans.csv", "Small-LI": "LI-Small_Trans.csv"}
PATTERNS = {"Small-HI": "HI-Small_Patterns.txt", "Small-LI": "LI-Small_Patterns.txt"}
RAW = {}
for name, fname in DATASETS.items():
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if not matches:
        print(f"PERINGATAN: {fname} tidak ditemukan di /kaggle/input")
        continue
    RAW[name] = matches[0]
    raw_dir = f"{DATA_DIR}/AML/{name}/raw"
    os.makedirs(raw_dir, exist_ok=True)
    if not os.path.exists(f"{raw_dir}/{fname}"):
        os.symlink(matches[0], f"{raw_dir}/{fname}")
    print(f"{name}: {matches[0]}")

## 2b. Metadata untuk analisis (sekali per dataset, beberapa menit)

`thesis/metadata.py` membuat `meta_<dataset>.csv.gz` (satu baris per transaksi mentah): split, periode **aktif** (<= 10 Sep 2022) / **ekor**, tipe pola dari `*_Patterns.txt`, dan Î”t kausal. Skor per transaksi dari tiap run (`scores_test.npz`) nanti digabung lewat `edge_id`.

Angka acuan (dihitung terpisah dengan awk dari CSV mentah) -- output sel ini harus sama:

| | HI-Small | LI-Small |
|---|---|---|
| Hari train / val / test | 0-5 / 6-7 / 8-17 | 0-5 / 6-7 / 8-16 |
| Transaksi train / val / test | 3.248.921 / 965.524 / 863.900 | 4.432.934 / 1.316.442 / 1.174.673 |
| Ilegal train / val / test | 2.530 / 1.036 / 1.611 | 1.993 / 770 / 802 |
| Test periode ekor: transaksi / ilegal | 1.108 / 655 (40,7% ilegal test) | 223 / 134 (16,7% ilegal test) |
| Transaksi ilegal yang masuk percobaan berpola | 3.209 dari 5.177 | 1.023 dari 3.565 |
| Pengirim punya transaksi masuk sebelumnya (Î”t >= 0) | 86,60% | 86,42% |

In [ ]:
for name in DATASETS:
    out = f"{OUT_DIR}/meta_{name}.csv.gz"
    if os.path.exists(out) or name not in RAW:
        continue
    pat = glob.glob(f"/kaggle/input/**/{PATTERNS[name]}", recursive=True)
    if not pat:
        print(f"PERINGATAN: {PATTERNS[name]} tidak ditemukan di /kaggle/input")
        continue
    print(f"===== {name}")
    !cd {REPO_DIR} && python thesis/metadata.py --trans "{RAW[name]}" --patterns "{pat[0]}" --out "{out}"

## 3. Helper

`run_thesis(model, dataset, seed)` menjalankan `configs/thesis/AML-Small-<HI|LI>-<Multi|MEGA>-GIN.yaml`. Run yang sudah punya `run_summary.json` di `OUT_DIR` dilewati, jadi sel bisa dijalankan ulang tanpa mengulang run yang selesai.

In [ ]:
import json, subprocess, time
import numpy as np
import pandas as pd

TAG = "thesis"
PAPER_F1 = {  # Bilgi & Atasu (TMLR 2026), Tabel 16: F1 kelas ilegal (%), rerata dan simpangan baku
    ("Multi-GIN", "Small-HI"): (62.66, 1.73), ("Multi-GIN", "Small-LI"): (32.21, 0.99),
    ("MEGA-GIN", "Small-HI"): (70.83, 2.19), ("MEGA-GIN", "Small-LI"): (43.67, 0.55),
}


def cfg_path(model, dataset):
    return f"configs/thesis/AML-{dataset}-{model}.yaml"


def run_dir_of(model, dataset, seed):
    # MEGA menamai folder run <nama-config>-<name_tag>-gpu<idx>/<seed>.
    return f"{OUT_DIR}/AML-{dataset}-{model}-{TAG}-gpu0/{seed}"


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def summarize(model, dataset, seed):
    rd = run_dir_of(model, dataset, seed)
    s = json.load(open(f"{rd}/run_summary.json"))
    val_f1_curve = [round(r["f1"] * 100, 2) for r in read_jsonl(f"{rd}/val/stats.json")]
    return {
        "model": model, "dataset": dataset, "seed": seed,
        "epochs_run": s["epochs_run"], "best_epoch": s["best_epoch"], "early_stop": s["stopped_early"],
        "val_f1": s["val"]["f1"] * 100, "test_f1": s["test"]["f1"] * 100,
        "test_precision": s["test"]["precision"] * 100, "test_recall": s["test"]["recall"] * 100,
        "test_pr_auc": s["test"]["pr-auc"] * 100,
        "paper_f1": PAPER_F1[(model, dataset)][0], "paper_std": PAPER_F1[(model, dataset)][1],
        "train_epoch_min": np.mean(s["train_epoch_s"]) / 60, "val_eval_min": np.mean(s["val_eval_s"]) / 60,
        "test_eval_min": s["test_eval_s"] / 60,
        "run_jam": (sum(s["train_epoch_s"]) + sum(s["val_eval_s"]) + s["test_eval_s"]) / 3600,
        "gpu_mem_mb": s["peak_gpu_mem_reserved_mb"], "params": s["params"],
        "test_cakupan_%": 100 * s["n_test_pred_edges"] / s["n_seed_edges"]["test"],
        "val_f1_per_epoch": val_f1_curve,
    }


def run_thesis(model, dataset, seed):
    rd = run_dir_of(model, dataset, seed)
    if os.path.exists(f"{rd}/run_summary.json"):
        print(f"Lewati {model}/{dataset}/seed {seed}: sudah selesai ({rd})")
        return summarize(model, dataset, seed)
    cmd = ["python", "-u", "-m", "MegaGNN.main", "--cfg", cfg_path(model, dataset), "--gpu", "0",
           "seed", str(seed), "out_dir", OUT_DIR, "dataset.dir", DATA_DIR, "name_tag", TAG]
    print("=" * 80 + "\n" + " ".join(cmd) + "\n" + "=" * 80)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Run gagal (exit {proc.returncode}) -- lihat log di atas / {rd}/logging.log")
    s = summarize(model, dataset, seed)
    print(f"\nSelesai {(time.time() - t0) / 3600:.2f} jam | {s['epochs_run']} epoch (terbaik {s['best_epoch']}) | "
          f"test F1 {s['test_f1']:.2f}% (paper {s['paper_f1']:.2f} +- {s['paper_std']:.2f})")
    return s

## 4. Jalankan run

Satu dataset per sesi commit. Urutan: Multi-GIN lalu MEGA-GIN.

In [ ]:
DATASET = "Small-HI"   # sesi berikutnya: "Small-LI"
SEED = 1

results = [run_thesis(m, DATASET, SEED) for m in ["Multi-GIN", "MEGA-GIN"]]

## 5. Ringkasan

In [ ]:
done = []
for m in ["Multi-GIN", "MEGA-GIN"]:
    for d in DATASETS:
        for rs in sorted(glob.glob(f"{OUT_DIR}/AML-{d}-{m}-{TAG}-gpu0/*/run_summary.json")):
            done.append(summarize(m, d, int(rs.split("/")[-2])))

if done:
    res = pd.DataFrame(done)
    cols = ["model", "dataset", "seed", "epochs_run", "best_epoch", "early_stop", "val_f1", "test_f1", "paper_f1",
            "paper_std", "test_precision", "test_recall", "test_pr_auc", "train_epoch_min", "val_eval_min",
            "run_jam", "gpu_mem_mb", "test_cakupan_%"]
    display(res[cols].round(2))
    print("\nF1 val per epoch (%):")
    for r in done:
        print(f"  {r['model']:9s} {r['dataset']}: {r['val_f1_per_epoch']}")

# Paket hasil untuk diunduh (tanpa checkpoint, agar kecil).
!cd /kaggle/working && rm -f results_thesis.zip && zip -qr results_thesis.zip results_thesis -x "*/ckpt/*" && ls -lh results_thesis.zip